Generador masivo de Asunto + Cuerpo desde memorandums PDF
 CONFIRMADO localmente (probado, no asumido):
   - Regex de campos principales: contra 1 PDF real (siniestro 231148619).
   - Regex de MEM / Planilla: contra 2 fragmentos de texto.
   - Negrita / cursiva+gris: persisten al guardar/releer.
   - Estilos (header pintado, bordes, anchos, freeze panes):
     probados junto con celdas rich-text, no rompen el formato.

 NO CONFIRMADO — sigue pendiente, no se resolvió en esta iteración:
   - No he visto la salida de correr el batch completo de 20 PDFs.
   - Colores de marca: no me diste hex, usé un azul gris neutro (2F5496)
     como default arbitrario. Si necesitas el color corporativo de
     La Positiva, dame el código hex y lo cambio.


-!pip install pypdf openpyxl --quiet

In [1]:
!pip install pypdf openpyxl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.1 MB/s eta 0:00:00


In [4]:
try:
    import pypdf
    print("pypdf:", pypdf.__version__)
except ImportError:
    print("pypdf NO instalado")

try:
    import openpyxl
    print("openpyxl:", openpyxl.__version__)
except ImportError:
    print("openpyxl NO instalado")

try:
    import pandas as pd
    print("pandas:", pd.__version__)
except ImportError:
    print("pandas NO instalado")

pypdf: 6.14.2
openpyxl: 3.1.5
pandas: 2.2.2


In [6]:
import re
import glob
import os
from datetime import datetime
from pypdf import PdfReader

CARPETA_PDFS = "/content/drive/MyDrive/1. LP_PROYECTOS/GENERAR_ASUNTO_CONTENIDO_CORREO/MEMOS"
CARPETA_SALIDA = "/content/drive/MyDrive/1. LP_PROYECTOS/GENERAR_ASUNTO_CONTENIDO_CORREO"

PATRONES = {
    'siniestro':    r'Siniestro:\s*(\d+)',
    'personas':     r'N[uú]mero de personas:\s*(\d+)',
    'monto':        r'Monto de la indemnizaci[oó]n de esta planilla:\s*S/\s*([\d,]+\.\d{2})',
    'cultivos':     r'Cultivos indemnizados:\s*(.+?)\.',
    'departamento': r'Departamento:\s*(.+?)\.',
    'provincia':    r'Provincia:\s*(.+?)\.',
    'poliza':       r'P[oó]liza:\s*(\d+)',
    'campania':     r'Catastr[oó]fico\s+(\d{4}-\d{4})',
    'mem':          r'MEMOR[AÁ]NDUM\s+Seg\.Rurales-(\d+-\d{4})',
    'planilla':     r'Planilla\s+(\d+)\s+del\s+Siniestro',
}

CAMPOS_OBLIGATORIOS = ['siniestro', 'departamento', 'campania', 'personas', 'monto']
CAMPOS_NOTA = ['mem', 'planilla']

COLOR_NOTA_GRIS = "#BFBFBF"


def extraer_campos(texto: str) -> dict:
    campos = {}
    for nombre, patron in PATRONES.items():
        m = re.search(patron, texto)
        campos[nombre] = m.group(1).strip() if m else None
    return campos


def armar_asunto(campos: dict) -> str:
    return (f"SINIESTRO {campos['siniestro']} - "
            f"{campos['departamento'].upper()} {campos['campania']} / "
            f"{campos['personas']} p")


def armar_bloque_html(campos: dict) -> str:
    asunto = armar_asunto(campos)
    nota_html = ""
    if all(campos.get(k) for k in CAMPOS_NOTA):
        nota_html = (f"<br><br><i style='color:{COLOR_NOTA_GRIS}'>Nota interna: "
                     f"MEM ({campos['mem']}) - Planilla {campos['planilla']}</i>")
    return (
        f"<div class='bloque'>"
        f"<p><b>{asunto}</b></p>"
        f"<p>Estimada Catherine buen día,<br><br>"
        f"Envío adjunto el archivo para que por favor generen la planilla respectiva.<br><br>"
        f"El monto es de S/ <b>{campos['monto']}</b>{nota_html}</p>"
        f"</div>"
    )


# =========================================================
# PASO 1 — Contar y validar apertura de PDFs
# =========================================================
pdfs = sorted(glob.glob(f'{CARPETA_PDFS}/*.pdf') + glob.glob(f'{CARPETA_PDFS}/*.PDF'))
print(f"PDFs encontrados: {len(pdfs)}\n")

if len(pdfs) == 0:
    print("⚠ 0 PDFs. Revisa CARPETA_PDFS con:")
    print(f"   os.path.exists('{CARPETA_PDFS}') ->", os.path.exists(CARPETA_PDFS))

fecha_ejecucion = datetime.now().strftime("%Y%m%d_%H%M")
nombre_base = f"correos_masivos_{len(pdfs)}pdf_{fecha_ejecucion}"
SALIDA_HTML = f"{CARPETA_SALIDA}/{nombre_base}.html"

pdfs_ok = []
for ruta in pdfs:
    nombre = ruta.split('/')[-1]
    try:
        reader = PdfReader(ruta)
        if len(reader.pages) == 0:
            print(f"✗ {nombre}: 0 páginas")
            continue
        _ = reader.pages[0].extract_text()
        pdfs_ok.append(ruta)
    except Exception as e:
        print(f"✗ {nombre}: error al abrir -> {e}")

print(f"\n✓ Abren correctamente: {len(pdfs_ok)} de {len(pdfs)}")
print("--- Revisa lo anterior antes de seguir ---\n")


# =========================================================
# PASO 2 — Extraer y generar HTML
# =========================================================
html_partes = ["<html><head><meta charset='utf-8'>"
               "<style>"
               "body{font-family:Calibri,Arial,sans-serif;font-size:14px;color:#333;"
               "max-width:700px;margin:20px auto;line-height:1.5}"
               ".bloque{padding:16px 20px;margin-bottom:18px;border:1px solid #E0E0E0;"
               "border-radius:6px;background:#FAFAFA}"
               ".bloque p{margin:0 0 10px 0}"
               ".bloque b{color:#2F5496}"
               "</style></head><body>"]

id_actual = 0
n_revisar = 0
n_error = 0
n_sin_nota = 0

for ruta in pdfs_ok:
    nombre = ruta.split('/')[-1]
    id_actual += 1
    try:
        texto = PdfReader(ruta).pages[0].extract_text()
        campos = extraer_campos(texto)
        faltantes = [k for k in CAMPOS_OBLIGATORIOS if not campos.get(k)]

        if faltantes:
            n_revisar += 1
            print(f"⚠ id {id_actual} ({nombre}): faltan {faltantes}")
            html_partes.append(f"<p style='color:red'>[REVISAR {nombre}: faltan {faltantes}]</p><hr>")
            continue

        if not all(campos.get(k) for k in CAMPOS_NOTA):
            n_sin_nota += 1
            print(f"ℹ id {id_actual} ({nombre}): sin MEM/Planilla -> nota omitida")

        html_partes.append(armar_bloque_html(campos))

    except Exception as e:
        n_error += 1
        print(f"✗ id {id_actual} ({nombre}): error -> {e}")
        html_partes.append(f"<p style='color:red'>[ERROR {nombre}: {e}]</p><hr>")

html_partes.append("</body></html>")

with open(SALIDA_HTML, "w", encoding="utf-8") as f:
    f.write("\n".join(html_partes))

print(f"\n=== RESUMEN ===")
print(f"Total procesados:          {id_actual}")
print(f"Con campos faltantes:      {n_revisar}")
print(f"Con error:                 {n_error}")
print(f"OK sin nota (falta MEM):   {n_sin_nota}")
print(f"OK completos:              {id_actual - n_revisar - n_error}")
print(f"\nHTML: {SALIDA_HTML}")

PDFs encontrados: 10


✓ Abren correctamente: 10 de 10
--- Revisa lo anterior antes de seguir ---


=== RESUMEN ===
Total procesados:          10
Con campos faltantes:      0
Con error:                 0
OK sin nota (falta MEM):   0
OK completos:              10

HTML: /content/drive/MyDrive/1. LP_PROYECTOS/GENERAR_ASUNTO_CONTENIDO_CORREO/correos_masivos_10pdf_20260709_2304.html
